# MT3：Transformer 多目标跟踪实验

本 Notebook 将作者的训练与评估入口整理为可逐格运行的实验，模型、损失和运动/量测生成代码直接来自同目录的 `MT3/src/`。

**论文：** Juliano Pinto, Georg Hess, William Ljungbergh, Yuxuan Xia, Lennart Svensson, Henk Wymeersch.
*Next Generation Multitarget Trackers: Random Finite Set Methods vs Transformer-based Deep Learning*.
2021. [arXiv:2104.00734](https://arxiv.org/abs/2104.00734).

**原代码：** [JulianoLagana/MT3](https://github.com/JulianoLagana/MT3)，
转换基于提交 `c1270e1de5b8d68eab50a797d16061310fa95d97`。原作者署名与 MIT 许可保留于 `MT3/LICENSE`。

默认运行 `quick` 短程实验：原模型架构、Task 1、20 次优化、批量2、8个独立测试窗口。
这用于学习和验证流程，不能当作论文表格结果。仓库没有附带已训练权重；默认从随机权重开始训练。


### 方法机理：从量测集合到目标集合

本目录实现 **MT3（Transformer 多目标跟踪）实验**。论文是 Pinto 等人的 *Next Generation Multitarget Trackers: Random Finite Set Methods vs Transformer-based Deep Learning*（2021），[论文原文](https://arxiv.org/html/2104.00734v3)。下述细节以固定提交的作者源码和当前 YAML 为准；公式是对代码的整理，未复刻论文版面。

### 1. 输入与输出：从量测集合预测目标集合

每个样本包含20个时刻、间隔0.1的二维量测，单条记录为 `[x, y, t]`。不同时间的点合成一个带时间标记的集合；同一时刻可能有多个目标的量测、杂波，也可能漏检。网络输出窗口末时刻的16个候选位置及存在概率，阈值筛选后形成目标集合。

20个时刻对应 `t=0,...,1.9`，不是20次优化；训练一次优化使用一批新生成的完整窗口。本实现标签仅含末时刻位置，不输出速度或跨窗口轨迹ID。测试窗口彼此独立，不能把这里的散点图解释成连续身份跟踪结果。

代码入口：[量测生成](MT3/src/data_generation/mot_data_generation.py)、[批量生成与标签](MT3/src/data_generation/data_generator.py)。真实ID用于生成监督信号和 oracle；虽然 `MOTT.forward` 接收 `unique_ids` 参数，前向特征并未使用它。

### 2. Encoder：让量测相互提供上下文

位置先除以视场宽度20，再经线性层映射到256维；时间 `round(t/dt)` 通过可学习时间嵌入编码。Padding 被 mask。6层 Encoder、每层8头注意力，使一个点能够结合其他时刻、其他位置的证据判断其含义。

下面用“每行一个 token”的记法表示单头注意力，省略多头拼接、残差、LayerNorm和前馈层：

$$
A(Q,K,V)=\operatorname{softmax}_{\mathrm{row}}\!\left(\frac{QK^{\mathsf T}}{\sqrt{d_k}}\right)V
$$

Q为查询、K为被比较的键、V为汇聚的值。匹配得分经 softmax 变成权重，再加权汇聚V。网络通过训练学习哪些证据相关；注意力权重本身不是经过概率校准的数据关联后验。Encoder 的自注意力允许不同量测交换信息，不需要先手工分配量测所属目标。

代码：[PreProccessor 与 Transformer 各层](MT3/src/modules/models/mt3/transformer.py)、[时间编码与模型前向](MT3/src/modules/models/mt3/mt3.py)。这里位置预处理为除以20；proposal 分支另加0.5，两个步骤不能混为一谈。

### 3. Two-stage proposal、Decoder 与迭代位置细化

Encoder 给量测特征预测候选位置偏移和目标分数，选取分数最高的16项作为初始化 proposal。位置编码经变换得到 Decoder 的查询位置和初始内容。默认 `two_stage=True` 时，查询由输入相关 proposal 初始化；不能简单理解为16个固定、独立的目标ID。

6层 Decoder 同时进行候选间自注意力和候选对 Encoder memory 的交叉注意力，前者协调不同候选，后者从整个量测窗口取证。每层预测位置修正量；对归一化参考位置执行 logit 空间的残差更新，再映射回视场坐标：

$$
r_i^{(\ell+1)}=\sigma\!\left(\operatorname{logit}(r_i^{(\ell)})+\Delta_i^{(\ell)}\right),\qquad \hat{x}_i=20\left(r_i^{(L)}-\tfrac12\right)
$$

`r`为 `(0,1)` 内的参考位置，sigma为 sigmoid，Delta由当前位置特征的MLP产生。每个候选还输出一个 logit，sigmoid 后得到存在概率。评估严格保留 `probability > 0.9` 的候选。未过阈值的灰色候选不计为检测目标；16是槽位上限，并非假设真实目标数恒为16。

代码：[MOTT、proposal 与输出头](MT3/src/modules/models/mt3/mt3.py)、[Decoder 更新](MT3/src/modules/models/mt3/transformer.py)。窗口必须有足够的量测以执行原代码的 top-k=16；Notebook 对预览批次做了检查。

### 4. Hungarian 集合匹配：解决目标无固定排列的问题

真实目标没有天然的“第1个、第2个”顺序。直接逐索引回归，会因标签排列任意而惩罚正确的集合。作者代码先对16个输出槽与真实目标构造代价矩阵，用 Hungarian 算法找一对一匹配。当前 `order=1` 的匹配代价是二维欧氏距离减去该槽存在概率：

$$
\hat{\pi}=\arg\min_{\pi\;\mathrm{injective}}\sum_{j=1}^{n}\!\left(\lVert \hat{x}_{\pi(j)}-y_j\rVert_2-\hat{p}_{\pi(j)}\right)
$$

其中n是真实目标数，pi把每个真实目标匹配到不同的预测槽。找到匹配后，匹配槽的存在标签为1，其余为0；匹配位置用 **L1回归**，存在性用带 `pos_weight` 的 BCEWithLogitsLoss。权重由 `(16-avg_gt_objects)/avg_gt_objects` 给出。匹配阶段的L2距离与训练回归阶段的L1损失是不同步骤。

默认损失包含最终输出、5层中间 Decoder 输出及 Encoder proposal 输出，并附加4倍对比损失：

$$
\mathcal{L}=\mathcal{L}^{\mathrm{final}}_{\mathrm{DETR}}+\sum_{\ell=1}^{5}\mathcal{L}^{(\ell)}_{\mathrm{DETR}}+\mathcal{L}^{\mathrm{enc}}_{\mathrm{DETR}}+4\mathcal{L}_{\mathrm{contrastive}}
$$

代码：[compute_hungarian_matching / detr_forward](MT3/src/modules/loss.py)、[原训练入口](MT3/src/training.py)、[模型配置](MT3/configs/models/mt3.yaml)。默认是 DETR 损失训练，不是直接对阈值化的 GOSPA 反向传播。

### 5. 对比学习：让同源量测的表示更相近

Encoder 特征经无偏置线性投影并归一化到单位球面，用点积比较不同量测。屏蔽自身与 padding 后，按行做 log-softmax。训练时利用仿真ID，把同一目标的其他量测分配为正样本，对这些正样本的负对数相似概率求平均。

这样，网络除了学“最终位置在哪、是否存在”，还得到“哪些点来自同一对象”的辅助监督。监督ID只在训练损失里使用。原实现将杂波ID统一标为 `-1`，因此杂波之间也会进入同ID正样本组；并非每个杂波有独立身份。没有其他同源量测的行不贡献有效对比项。

代码：[ContrastiveClassifier](MT3/src/modules/contrastive_classifier.py)、[ContrastiveLoss](MT3/src/modules/contrastive_loss.py)。默认关闭额外的 `false_classifier`。

### 6. GOSPA：同时惩罚位置、漏检和虚警

本实验采用 `p=1, c=2, alpha=2` 的 GOSPA。X是真实位置集合，Xhat是通过存在阈值后的预测集合，gamma是不要求覆盖全部元素的一对一部分匹配：

$$
D(X,\hat{X})=\min_{\gamma}\left[\sum_{(i,j)\in\gamma}\lVert x_i-\hat{x}_j\rVert_2+\frac{c}{2}\bigl(|X|+|\hat{X}|-2|\gamma|\bigr)\right],\qquad p=1,\ \alpha=2,\ c=2
$$

部分匹配允许位置相差过大的两个点分别记作一次漏检与一次虚警。最优解可以选择仅匹配距离小于c的点。由此分为匹配位置距离之和、每个未匹配真目标的 `c/2=1` 代价、每个未匹配预测的 `c/2=1` 代价。报告中的 localization 是总定位代价，不除以匹配数，三项能直接相加。

例如有3个真目标却没有任何预测，则GOSPA=3，定位项=0、漏检项=3、虚警项=0。定位项为0此时意味着没有匹配，不能据此说定位准确。

代码：[原始GOSPA与分解](MT3/src/modules/loss.py)、[评估入口](MT3/src/modules/evaluator.py)、[评估参数](MT3/configs/eval/default.yaml)。GOSPA不按目标数归一化；根目录BP实验的OSPA采用另一组定义、参数和场景，两者不能直接比较原始数值。Prob-GOSPA使用软存在概率、DETR指标对应训练损失，也不应与GOSPA横向比较大小。

### 7. 与 BP 实验的关系和适用边界

根目录BP实验按运动/量测模型传播粒子分布，并以置信传播近似求解数据关联。MT3则利用合成数据训练注意力网络，以固定窗口的量测集合端到端输出目标集合。注意力汇聚可直观理解为特征交换，但本实现没有执行BP的消息更新方程。

本次短程实验用于检查数据→前向→损失→梯度→权重保存/重载→独立评估链路。预测泛化能力依赖训练规模和场景分布，当前结果不足以判断MT3是否优于BP或论文中的RFS基线。参数差异与真实测试结果见[实验报告](docs/EXPERIMENT_REPORT.md)。



## 1. 环境与运行方法

在 VS Code 中打开包含本文件和 `MT3/` 子文件夹的项目目录，选择已安装依赖的 Python 内核，再点击“全部运行”。
本次验证使用 Python 3.12；依赖列在 `requirements-notebook.txt`。需要新建环境时，在项目终端执行：

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements-notebook.txt
```

Windows 的激活命令为 `.venv\Scripts\Activate.ps1`。若提示找不到 `torch`，请确认 Notebook 右上角内核与安装依赖的解释器相同。
本文件调用同目录的原始模块，需要保留 `MT3/` 文件夹，不是独立于源码的单文件实现。

In [ ]:
import copy
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import random
import sys
import time
from datetime import datetime, timezone

required = {'torch': 'torch', 'numpy': 'numpy', 'scipy': 'scipy',
            'matplotlib': 'matplotlib', 'pandas': 'pandas', 'yaml': 'PyYAML'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('缺少依赖：' + ', '.join(missing)
                       + '。请在当前内核环境安装 requirements-notebook.txt。')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

anchors = [Path.cwd(), *Path.cwd().parents]
candidates = [candidate for anchor in anchors
              for candidate in (anchor, anchor / 'experiments/mt3')]
ROOT = next((p for p in candidates
             if (p / 'MT3/src/modules/models/mt3/mt3.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('请在包含 MT3/ 子目录的项目文件夹启动 Notebook。')
SOURCE_ROOT = ROOT / 'MT3'
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE_ROOT / 'src'))

from util.load_config_files import load_yaml_into_dotdict
from data_generation.data_generator import DataGenerator
from data_generation.mot_data_generation import MotDataGenerator
from modules.models.mt3.mt3 import MOTT
from modules.loss import MotLoss, FalseMeasurementLoss
from modules.contrastive_loss import ContrastiveLoss
from modules import evaluator

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
print('Python:', platform.python_version(), '| PyTorch:', torch.__version__)
print('Project:', ROOT)

## 2. 实验配置

- `TASK='task1'`：无量测噪声、无杂波、无随机出生/死亡，目标仍可能离开视场。
- `TASK='task2'`：有运动噪声、量测噪声、漏检、杂波和目标出生/死亡，计算量更大。
- `PROFILE='quick'`：20次优化、batch=2、8个测试窗口；**模型架构不缩小**。
- `PROFILE='full'`：使用原 YAML 的训练参数（1000000次优化、batch=32），评估1000个窗口，适合长期训练。**full仅表示作者YAML原配置，不等于严格复现论文**：任务分布及训练步数有差异，见下文及实验报告。

`TRAIN_STEPS_OVERRIDE` 等选项可以单独覆盖步数。已有权重时设置 `CHECKPOINT_PATH`；仅推理请同时设置 `TRAIN_STEPS_OVERRIDE=0`。
每次从头运行创建一个新的结果文件夹。`RESUME_TRAINING=True` 还会恢复优化器/调度器；数据流 RNG 状态未保存，恢复不保证与未中断训练逐位一致。

### 当前配置与论文的差异

这里的Task 1/2指作者仓库中的文件名，不保证与论文同名场景一致。原`task1.yaml`采用探测率1、杂波0、运动与量测噪声0、随机出生/死亡0；论文Task 1含噪声、漏检、杂波和出生/死亡。Task 2虽启用这些因素，出生分布、噪声参数实现等仍需逐项核对。原YAML的训练步数为100万，论文报告Task 1为80万、Task 2为60万。

数据生成中位置出生采用视场内均匀分布，`p_add`用作Poisson出生数均值，`sigma_q`直接乘入过程噪声协方差。Notebook保留这些作者实现，不静默改成论文公式。生成器会重采样直到末时刻真值非空，因此这里不测试整个窗口末时刻没有目标的情况。

完整配置与差异表见[实验报告](docs/EXPERIMENT_REPORT.md)。需要论文级复现时，应先对齐数据分布、训练预算、评估与基线，不能仅把`PROFILE`设为`full`。

In [ ]:
TASK = 'task1'                     # 'task1' 或 'task2'
PROFILE = 'quick'                  # 'quick' 或 'full'
DEVICE = 'auto'                    # 'auto' / 'cpu' / 'cuda'；auto 与作者一致
SEED = 0
EVAL_SEED = 1
TRAIN_STEPS_OVERRIDE = None        # 例如 2000；None 使用所选 PROFILE
BATCH_SIZE_OVERRIDE = None
EVAL_WINDOWS_OVERRIDE = None
CHECKPOINT_PATH = None             # 例如 ROOT / 'results_mt3/.../checkpoint_final.pt'
RESUME_TRAINING = False

if TASK not in {'task1', 'task2'} or PROFILE not in {'quick', 'full'}:
    raise ValueError('请检查 TASK 和 PROFILE。')
params = load_yaml_into_dotdict(str(SOURCE_ROOT / f'configs/tasks/{TASK}.yaml'))
params.update(load_yaml_into_dotdict(str(SOURCE_ROOT / 'configs/models/mt3.yaml')))
original_training = copy.deepcopy(params.training)
params.data_generation.seed = SEED
if PROFILE == 'quick':
    params.training.n_gradient_steps = 20
    params.training.batch_size = 2
if TRAIN_STEPS_OVERRIDE is not None:
    params.training.n_gradient_steps = int(TRAIN_STEPS_OVERRIDE)
if BATCH_SIZE_OVERRIDE is not None:
    params.training.batch_size = int(BATCH_SIZE_OVERRIDE)
N_EVAL = int(EVAL_WINDOWS_OVERRIDE) if EVAL_WINDOWS_OVERRIDE is not None else (8 if PROFILE == 'quick' else 1000)
if params.training.n_gradient_steps < 0 or params.training.batch_size < 1 or N_EVAL < 1:
    raise ValueError('训练步数必须非负，批量与评估窗口数必须为正。')
if DEVICE not in {'auto', 'cpu', 'cuda'}:
    raise ValueError('DEVICE 请选择 auto / cpu / cuda。')
params.training.device = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE
if params.training.device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('当前内核没有可用 CUDA，请设置 DEVICE="cpu"。')
device = torch.device(params.training.device)
torch.set_num_threads(min(4, os.cpu_count() or 1))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

run_name = f'{TASK}_{PROFILE}_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}'
OUTPUT = ROOT / 'results_mt3' / run_name
OUTPUT.mkdir(parents=True, exist_ok=False)
versions = {name: importlib.metadata.version(name)
            for name in ['torch', 'numpy', 'scipy', 'matplotlib', 'pandas', 'PyYAML']}
(OUTPUT / 'config_training.json').write_text(json.dumps(params, indent=2), encoding='utf-8')
display(pd.DataFrame([
    ['Task / profile', f'{TASK} / {PROFILE}'],
    ['Device', str(device)], ['Window timesteps', params.data_generation.n_timesteps],
    ['dt', params.data_generation.dt], ['Encoder / decoder layers', f'{params.arch.encoder.n_layers} / {params.arch.decoder.n_layers}'],
    ['d_model / queries', f'{params.arch.d_model} / {params.arch.num_queries}'],
    ['Optimizer steps', params.training.n_gradient_steps], ['Batch size', params.training.batch_size],
    ['Evaluation windows', N_EVAL]], columns=['Setting', 'Value']))
print('Results:', OUTPUT)

## 3. 在线合成数据与 Notebook 适配

一个样本是20个时间步的量测集合，每个量测含 `[x, y, t]`。标签是窗口**末时刻**仍存在目标的二维位置。
优化步数不等于帧数，8个测试窗口也不是一条160帧连续轨迹。

作者的 `DataGenerator` 默认启动 `multiprocessing.Pool()`。下面只将批量生成改为顺序执行，
保留其独立 RNG 的初始化、`get_batch()`、运动模型、量测模型、padding 和监督标签生成方式。
`unique_ids` 中 `-1` 是杂波，`-2` 是 padding；真实ID用于对比学习监督和 oracle 参考，不作为网络量测特征。

In [ ]:
class SerialPool:
    """Use the author's starmap interface without creating worker processes."""
    def starmap(self, function, arguments):
        return [function(*argument) for argument in arguments]

class NotebookDataGenerator(DataGenerator):
    """Original get_batch(), serial execution; same per-sample RNG initialization."""
    def __init__(self, config):
        self.params = config
        assert 0 <= config.data_generation.n_prediction_lag <= config.data_generation.n_timesteps
        self.device = config.training.device
        self.n_timesteps = config.data_generation.n_timesteps
        self.pool = SerialPool()
        seeds = np.random.SeedSequence(config.data_generation.seed).spawn(config.training.batch_size)
        self.datagens = [MotDataGenerator(config, rng=np.random.default_rng(seed)) for seed in seeds]

preview_generator = NotebookDataGenerator(params)
preview_batch, preview_labels, preview_ids, preview_trajectories = preview_generator.get_batch()
if preview_batch.tensors.shape[1] < params.arch.num_queries:
    raise RuntimeError('量测总数不足以生成原模型的16个 two-stage 查询，请检查数据配置。')
print('Measurements [batch, padded_measurements, 3]:', tuple(preview_batch.tensors.shape))
print('Padding mask:', tuple(preview_batch.mask.shape))
print('Ground-truth counts:', [len(y) for y in preview_labels])
assert preview_batch.mask.dtype == torch.bool
assert torch.isfinite(preview_batch.tensors).all()

## 4. 原始 MT3 模型和损失

使用作者的 `MOTT` 类（仓库中的类名）构建 MT3，包括 two-stage proposal、迭代位置细化、辅助输出和对比学习。
训练损失为所有启用的 DETR 输出损失之和，再加 `4 × contrastive_loss`，与默认 `training.py` 一致。
优化器为 Adam，学习率 `5e-5`，调度器为 ReduceLROnPlateau。

兼容处理只涉及训练入口：不使用旧版调度器的 `verbose` 参数，不启动桌面绘图窗口；原模型与损失文件不修改。

In [ ]:
model = MOTT(params)
model.to(device)  # The upstream MOTT.to() does not return self.
mot_loss = MotLoss(params)
contrastive_loss = ContrastiveLoss(params)
false_loss = FalseMeasurementLoss(params) if params.loss.false_classifier else None
optimizer = torch.optim.Adam(model.parameters(), lr=params.training.learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=params.training.reduce_lr_patience,
    factor=params.training.reduce_lr_factor)
parameter_count = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {parameter_count:,}')
print('Architecture: original YAML; no reduced hidden dimension or layer count.')

start_step = 0
if CHECKPOINT_PATH is not None:
    checkpoint_path = Path(CHECKPOINT_PATH).expanduser()
    if not checkpoint_path.is_absolute():
        checkpoint_path = ROOT / checkpoint_path
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    if 'params' in checkpoint:
        if checkpoint['params']['arch'] != json.loads(json.dumps(params.arch)):
            raise ValueError('检查点模型架构与当前参数不同，请先使用匹配的配置。')
        if checkpoint['params']['data_generation'] != json.loads(json.dumps(params.data_generation)):
            raise ValueError('检查点任务配置与当前任务不同，请先使用匹配的 TASK 和 SEED。')
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    if RESUME_TRAINING:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_step = int(checkpoint.get('step', 0))
    print('Loaded checkpoint:', checkpoint_path.name, '| previous optimizer steps:', start_step)
elif RESUME_TRAINING:
    raise ValueError('恢复训练需要指定 CHECKPOINT_PATH。')

## 5. 训练与保存

每次优化使用新合成的数据。`total_loss` 包含辅助层，`detr_last` 仅为最后一层主损失，二者数值不相等。
短程训练的损失不必单调下降，低损失也不能代替独立测试。
按原配置每200000步保存中间检查点，并在本次训练结束时保存一个最终检查点。

In [ ]:
def save_notebook_checkpoint(path, completed_steps):
    payload = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'params': json.loads(json.dumps(params)),
        'step': int(completed_steps),
        'task': TASK, 'profile': PROFILE,
        'source_commit': 'c1270e1de5b8d68eab50a797d16061310fa95d97',
    }
    torch.save(payload, path)

train_generator = NotebookDataGenerator(params)
history = []
model.train()
train_start = time.perf_counter()
print_every = max(1, min(100, params.training.n_gradient_steps // 5))
for local_step in range(params.training.n_gradient_steps):
    batch, labels, unique_ids, trajectories = train_generator.get_batch()
    outputs, memory, aux, queries, attention = model(batch, unique_ids)
    loss_dict, indices = mot_loss(outputs, labels, loss_type=params.loss.type)
    if params.loss.type == 'both':
        gospa_weight = (local_step / params.training.n_gradient_steps) ** 3
        total_loss = sum(gospa_weight * value if 'gospa' in key else value
                         for key, value in loss_dict.items())
    else:
        total_loss = sum(loss_dict.values())
    c_loss = torch.zeros((), device=device)
    if params.loss.contrastive_classifier:
        c_loss = contrastive_loss(aux['contrastive_classifications'], unique_ids)
        total_loss = total_loss + params.loss.c_loss_multiplier * c_loss
    if params.loss.false_classifier:
        total_loss = total_loss + params.loss.f_loss_multiplier * false_loss(aux['false_classifications'], unique_ids)
    if not torch.isfinite(total_loss):
        raise FloatingPointError(f'Non-finite loss at optimizer step {local_step + 1}')
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    value = float(total_loss.detach().cpu())
    scheduler.step(value)
    global_step = start_step + local_step + 1
    history.append({'step': global_step, 'total_loss': value,
                    'detr_last': float(loss_dict['detr'].detach().cpu()) if 'detr' in loss_dict else None,
                    'contrastive_loss': float(c_loss.detach().cpu()),
                    'learning_rate': optimizer.param_groups[0]['lr'],
                    'elapsed_seconds': time.perf_counter() - train_start})
    if local_step == 0 or (local_step + 1) % print_every == 0:
        print(f'Step {global_step} | total={value:.4f} | contrastive={history[-1]["contrastive_loss"]:.4f}'
              f' | elapsed={history[-1]["elapsed_seconds"]:.1f}s')
    if global_step % params.training.checkpoint_interval == 0:
        save_notebook_checkpoint(OUTPUT / f'checkpoint_{global_step}.pt', global_step)

training_seconds = time.perf_counter() - train_start
completed_steps = start_step + params.training.n_gradient_steps
checkpoint_file = OUTPUT / 'checkpoint_final.pt'
save_notebook_checkpoint(checkpoint_file, completed_steps)
history_frame = pd.DataFrame(history, columns=['step', 'total_loss', 'detr_last',
                                               'contrastive_loss', 'learning_rate', 'elapsed_seconds'])
history_frame.to_csv(OUTPUT / 'training_history.csv', index=False)
print('Saved:', checkpoint_file.name, '| completed optimizer steps:', completed_steps)

In [ ]:
if not history_frame.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), constrained_layout=True)
    axes[0].plot(history_frame.step, history_frame.total_loss, label='Total training loss')
    axes[0].plot(history_frame.step, history_frame.detr_last, label='Final DETR loss')
    axes[0].set(xlabel='Optimizer step', ylabel='Loss', title=f'MT3 {PROFILE} training / {TASK}')
    axes[0].legend(fontsize=8)
    axes[1].plot(history_frame.step, history_frame.contrastive_loss, color='#d6604d')
    axes[1].set(xlabel='Optimizer step', ylabel='Contrastive loss', title='Contrastive supervision')
    for ax in axes:
        ax.grid(alpha=0.2)
    fig.savefig(OUTPUT / 'training_loss.png', dpi=160)
    plt.show()
else:
    print('0 optimizer steps: evaluating the loaded or randomly initialized model.')

## 6. 检查点重新加载检查

新建一个相同架构的模型并从刚保存的检查点加载权重，对同一批输入检查输出一致性。
这能验证保存/加载流程，不代表模型已经训练充分。

In [ ]:
model.eval()
with torch.no_grad():
    reference_output = model(preview_batch, preview_ids)[0]
saved_checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=True)
reloaded_model = MOTT(params)
reloaded_model.to(device)
reloaded_model.load_state_dict(saved_checkpoint['model_state_dict'], strict=True)
reloaded_model.eval()
with torch.no_grad():
    reloaded_output = reloaded_model(preview_batch, preview_ids)[0]
for key in ['state', 'logits']:
    torch.testing.assert_close(reference_output[key], reloaded_output[key], rtol=1e-6, atol=1e-6)
assert saved_checkpoint['step'] == completed_steps
del reloaded_model, reloaded_output, reference_output, saved_checkpoint
print('PASS: checkpoint reload preserves state and existence-logit outputs.')

## 7. 独立测试与 GOSPA

加载作者的 `configs/eval/default.yaml`：batch=1、随机种子1、$p=1$、$c=2$、$\alpha=2$、存在概率阈值0.9。
训练用的 DETR 配置与评估用的 GOSPA 配置分开保存，训练窗口不拿来充当测试窗口。

GOSPA 对匹配位置的误差与漏检/虚警一起计分，越小越好。本例 $p=1,\alpha=2$ 时，

$$\mathrm{GOSPA} = \text{localization} + \frac{c}{2}\,N_{\mathrm{missed}} + \frac{c}{2}\,N_{\mathrm{false}}.$$

它不按目标数量归一化，也不等于前一个 BP 实验中的 OSPA。这里评价窗口末时刻的二维目标集合，不计算跨窗口身份切换。
同时保留作者的 Prob-GOSPA 和最后一层 DETR 指标；这些指标的定义和尺度不同，不应互相比大小。

参考项沿用作者评估代码：
- `Random (upstream)`：其存在概率固定为0.5，低于评估阈值0.9，因此 GOSPA 实际对应空预测集合；不要据此声称优于强随机跟踪器。
- `True measurements (oracle)`：用真实ID从末时刻量测中筛出非杂波，属于知道真值关联的参考，实际部署无法直接得到。
- 本仓库不包含本实验所需的 PMBM/\(\delta\)-GLMB 对比实现，本 Notebook 不声称完成论文全部对比实验。

In [ ]:
eval_params = copy.deepcopy(params)
eval_params.recursive_update(load_yaml_into_dotdict(str(SOURCE_ROOT / 'configs/eval/default.yaml')))
eval_params.data_generation.seed = EVAL_SEED
evaluation_loss = MotLoss(eval_params)
eval_generator = NotebookDataGenerator(eval_params)
np.random.seed(EVAL_SEED)  # upstream random baseline uses NumPy's global RNG
(OUTPUT / 'config_evaluation.json').write_text(json.dumps(eval_params, indent=2), encoding='utf-8')
model.eval()
evaluation_rows = []
first_scene = None
eval_start = time.perf_counter()
with torch.no_grad():
    for window in range(N_EVAL):
        batch, labels, unique_ids, trajectories = eval_generator.get_batch()
        predictions = model(batch, unique_ids)[0]
        methods = {
            'MT3': predictions,
            'Random (upstream)': evaluator.create_random_baseline(eval_params),
            'True measurements (oracle)': evaluator.create_true_measurement_baseline(batch.tensors, unique_ids, eval_params),
        }
        for method, output in methods.items():
            gospa, decomposition = evaluator.compute_gospa(
                output, labels, evaluation_loss,
                existance_threshold=eval_params.loss.existence_prob_cutoff)
            prob_gospa, _ = evaluation_loss.gospa_forward(output, labels, probabilistic=True)
            detr, _ = evaluation_loss(output, labels, loss_type='detr')
            predicted_count = int((output['logits'][0].sigmoid() > eval_params.loss.existence_prob_cutoff).sum().cpu())
            row = {'window': window + 1, 'method': method,
                   'true_count': len(labels[0]), 'predicted_count': predicted_count,
                   'gospa': float(gospa.cpu()), 'prob_gospa': float(prob_gospa.cpu()),
                   'detr_last': float(detr['detr'].cpu()),
                   'localization': float(decomposition['localization']),
                   'missed': float(decomposition['missed']), 'false': float(decomposition['false'])}
            assert np.isfinite([row[k] for k in ['gospa', 'prob_gospa', 'detr_last', 'localization', 'missed', 'false']]).all()
            assert np.isclose(row['gospa'], row['localization'] + row['missed'] + row['false'], atol=1e-5)
            evaluation_rows.append(row)
        if first_scene is None:
            first_scene = {'measurements': batch.tensors[0][~batch.mask[0]].cpu().numpy(),
                           'truth': labels[0].cpu().numpy(),
                           'state': predictions['state'][0].cpu().numpy(),
                           'probability': predictions['logits'][0].sigmoid().cpu().numpy().ravel()}
        if window == 0 or (window + 1) % max(1, N_EVAL // 4) == 0:
            print(f'Evaluated {window + 1}/{N_EVAL} independent windows')
evaluation_seconds = time.perf_counter() - eval_start
evaluation_frame = pd.DataFrame(evaluation_rows)
evaluation_frame.to_csv(OUTPUT / 'evaluation_windows.csv', index=False)
evaluation_frame['count_abs_error'] = abs(evaluation_frame.predicted_count - evaluation_frame.true_count)
metrics = evaluation_frame.groupby('method', sort=False).agg(
    windows=('window', 'count'), gospa_mean=('gospa', 'mean'),
    gospa_std=('gospa', lambda x: x.std(ddof=0)),
    localization_mean=('localization', 'mean'), missed_mean=('missed', 'mean'),
    false_mean=('false', 'mean'), count_MAE=('count_abs_error', 'mean'),
    prob_gospa_mean=('prob_gospa', 'mean'), detr_last_mean=('detr_last', 'mean'))
metrics.to_csv(OUTPUT / 'evaluation_summary.csv')
display(metrics.round(4))
print('GOSPA standard deviation: population definition (ddof=0); sample size:', N_EVAL)

## 8. 结果可视化

第一张图显示各独立测试窗口的末时刻 GOSPA 分解，条形总高度就是平均 GOSPA。
第二张图显示第一个测试窗口：所有输入量测、真实末时刻位置、超过0.9阈值的预测，以及16个查询的存在概率。
灰色空心点仅用于查看未通过阈值的候选位置，不计入输出集合。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8), constrained_layout=True)
bottom = np.zeros(len(metrics))
for column, label, color in [('localization_mean', 'Localization', '#4393c3'),
                              ('missed_mean', 'Missed-target cost', '#f4a582'),
                              ('false_mean', 'False-target cost', '#b2182b')]:
    values = metrics[column].to_numpy()
    ax.bar(np.arange(len(metrics)), values, bottom=bottom, label=label, color=color)
    bottom += values
ax.set_xticks(np.arange(len(metrics)), ['MT3', 'Random (upstream)', 'True measurements\n(oracle)'])
ax.set(ylabel='Mean GOSPA (p=1, c=2, alpha=2)',
       title=f'{TASK} / {PROFILE}: {N_EVAL} independent evaluation windows')
ax.legend(fontsize=8)
fig.savefig(OUTPUT / 'gospa_decomposition.png', dpi=160)
plt.show()

scene = first_scene
threshold = eval_params.loss.existence_prob_cutoff
selected = scene['probability'] > threshold
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
scatter = axes[0].scatter(scene['measurements'][:, 0], scene['measurements'][:, 1],
                          c=scene['measurements'][:, 2], cmap='viridis', s=10, alpha=0.45)
fig.colorbar(scatter, ax=axes[0], label='Measurement time')
axes[0].scatter(scene['truth'][:, 0], scene['truth'][:, 1], marker='x', s=80,
                color='black', label='Final-time truth', zorder=4)
axes[0].scatter(scene['state'][~selected, 0], scene['state'][~selected, 1],
                facecolors='none', edgecolors='gray', s=30, label='Below-threshold queries')
axes[0].scatter(scene['state'][selected, 0], scene['state'][selected, 1],
                marker='+', s=110, color='#b2182b', label='MT3 output', zorder=5)
axes[0].set(xlabel='x', ylabel='y', title=f'Window 1: truth={len(scene["truth"])} / predicted={selected.sum()}',
            xlim=(params.data_generation.field_of_view_lb, params.data_generation.field_of_view_ub),
            ylim=(params.data_generation.field_of_view_lb, params.data_generation.field_of_view_ub))
axes[0].set_aspect('equal')
axes[0].legend(fontsize=7, loc='upper left')
axes[1].bar(np.arange(1, len(selected) + 1), scene['probability'], color='#4393c3')
axes[1].axhline(threshold, color='#b2182b', linestyle='--', label=f'Existence threshold = {threshold}')
axes[1].set(xlabel='Query index', ylabel='Existence probability', ylim=(0, 1.04), title='Candidate existence probabilities')
axes[1].legend(fontsize=8)
fig.savefig(OUTPUT / 'example_window.png', dpi=160)
plt.show()
np.savez_compressed(OUTPUT / 'example_window.npz', **scene)

## 9. 记录实验与理解结论

`quick` 结果只能说明当前代码、训练梯度、检查点与评估流程可运行，不能证明复现了论文的精度，也不能证明 MT3 优于 PMBM 或 δ-GLMB。
正式复现需要充分训练、独立测试、多种随机种子、匹配论文的场景和基线实现，并保存配置和权重。
相同种子在不同 PyTorch/NumPy 版本、硬件或设备上不保证逐位一致。

本 Notebook 显式保留原模型文件、默认结构、损失公式与任务 YAML；不调用作者的桌面 GUI。
如需提高训练量，修改第2节的参数后从头运行。若仅重新评估已保存模型，请设置检查点路径并把训练步数覆盖为0。

In [ ]:
summary = {
    'paper': 'Next Generation Multitarget Trackers: Random Finite Set Methods vs Transformer-based Deep Learning',
    'paper_url': 'https://arxiv.org/abs/2104.00734',
    'source_repository': 'https://github.com/JulianoLagana/MT3',
    'source_commit': 'c1270e1de5b8d68eab50a797d16061310fa95d97',
    'task': TASK, 'profile': PROFILE, 'seed': SEED, 'eval_seed': EVAL_SEED,
    'device': str(device), 'python': platform.python_version(), 'versions': versions,
    'parameter_count': int(parameter_count),
    'optimizer_steps_this_run': int(params.training.n_gradient_steps),
    'optimizer_steps_total': int(completed_steps),
    'batch_size': int(params.training.batch_size), 'evaluation_windows': N_EVAL,
    'training_seconds': training_seconds, 'evaluation_seconds': evaluation_seconds,
    'checkpoint_reload_verified': True,
    'gospa_parameters': {'p': eval_params.loss.order, 'c': eval_params.loss.cutoff_distance,
                         'alpha': eval_params.loss.alpha, 'existence_threshold': threshold},
    'metrics': json.loads(metrics.to_json(orient='index')),
    'scope': 'Short pipeline demonstration; not a completed paper-scale reproduction.' if PROFILE == 'quick'
             else 'Original YAML training profile; task distributions differ from the paper. Not a strict paper reproduction.',
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Finished:', OUTPUT.name)
print('Checkpoint, configs, CSV metrics, 3 figures, and summary saved.')
print(f'MT3 mean GOSPA: {metrics.loc["MT3", "gospa_mean"]:.4f}')
if PROFILE == 'quick':
    print('本次为短程流程验证，不作为论文精度结论。')

## 已保存的短程实验：如何解读结果

仓库[实验报告](docs/EXPERIMENT_REPORT.md)记录真实运行结果，逐窗口CSV和图片放在[examples/](examples/)中。本文件提交为空输出版，运行后显示你自己本次实验的结果，并写入新的`results_mt3/`目录。

| 已保存案例 | 优化步数 / batch / 测试窗口 | MT3平均GOSPA | 超过0.9阈值的预测 |
|---|---|---:|---|
| 作者仓库Task 1短程训练 | 20 / 2 / 8 | 2.5000 | 8个窗口全部为0 |
| 作者仓库Task 2额外流程检查 | 2 / 1 / 2 | 9.0000 | 2个窗口全部为0 |

两组GOSPA全部由漏检构成。模型从随机权重开始，仅做极少优化，当前输出不能支持“成功达到论文精度”或“MT3优于其他方法”的结论。两组训练预算和样本数量不同，不能用2.5与9直接判断任务性能差距。Oracle使用真实ID，是有额外真值信息的参考；原Random基线概率固定0.5，经0.9阈值后也为空集合。

完整方法说明见[方法文档](docs/METHOD.md)。使用本目录请引用Pinto等人的MT3论文并保留`MT3/LICENSE`；根目录Meyer等人2017年论文的引用适用于BP实验。